# 04 - COPY INTO and file-level idempotency

Prerequisites / execution context: Run 00. Creates a fresh isolated input path and target table each time the configuration below runs.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R09: COPY INTO tutorial](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/copy-into); [R08: Unity Catalog volumes](https://docs.databricks.com/aws/en/volumes/).


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


## 1. Create two immutable source files
A new experiment ID avoids combining an old target history with a fresh input location.


In [ ]:
import uuid
run_id=uuid.uuid4().hex[:10]
source=f"{VOLUME}/copy_{run_id}"
T=f"{BRONZE}.copy_{run_id}"
dbutils.fs.put(f"{source}/part1.json",'{"id":1,"amount":10}\n{"id":2,"amount":20}',True)
spark.sql(f"CREATE TABLE {T} (id BIGINT, amount BIGINT) USING DELTA")
cmd=f"COPY INTO {T} FROM '{source}' FILEFORMAT = JSON"
spark.sql(cmd).show()
assert spark.table(T).count()==2


## 2. Re-run without a new file
Expect the same two rows. The command tracks loaded files; it does not deduplicate business keys across different files.


In [ ]:
spark.sql(cmd).show()
assert spark.table(T).count()==2
dbutils.fs.put(f"{source}/part2.json",'{"id":2,"amount":20}\n{"id":3,"amount":30}',True)
spark.sql(cmd).show()
assert spark.table(T).count()==4
assert spark.table(T).select("id").distinct().count()==3


## Challenge and solution
Why does id 2 appear twice? It came from two different files. Apply business-key deduplication downstream. For an intentional reload, investigate COPY_OPTIONS force in the command reference before using it; reprocessing can duplicate records. Explain when Auto Loader is a better fit for file discovery at scale. Official Databricks documentation: [R10: Auto Loader](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader); [R12: Auto Loader file detection modes](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/file-detection-modes).
